<a href="https://colab.research.google.com/github/ryo-67/body-politic/blob/main/SentimentAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Sentiment Analysis

Sentiment Analysis of extracted transcripts

note: upload extracted_transcripts.xlsx

In [ ]:
# Install TextBlob if not already installed
%pip install textblob

In [ ]:
from textblob import TextBlob

# Define a function to classify sentiment based on TextBlob's polarity
def classify_sentiment(text):
    if pd.isna(text): # Handle NaN values in text column
        return 'neutral'
    analysis = TextBlob(str(text))
    polarity = analysis.sentiment.polarity
    subjectivity = analysis.sentiment.subjectivity

    # Adjusted Heuristic mapping for sentiment categories for news commentary
    if polarity > 0.3: # Strong positive
        return 'happy'
    elif polarity < -0.6 and subjectivity > 0.6: # Very strong negative, with high subjectivity (likely angry critique)
        return 'angry'
    elif polarity < -0.3: # General negative, but not as extreme/subjective as 'angry'
        return 'sad'
    else: # Neutral range, or less intense negative/positive
        return 'neutral'

# Apply sentiment analysis to the 'text' column
df['sentiment'] = df['text'].apply(classify_sentiment)

# Display the distribution of sentiments
sentiment_counts = df['sentiment'].value_counts()
print("\nSentiment Distribution (Adjusted Heuristic):")
print(sentiment_counts)

# Display some examples with their sentiment
print("\nExamples of classified sentiments (Adjusted Heuristic):")
display(df[['text', 'sentiment']].sample(10))


Sentiment Distribution:
sentiment
neutral    16235
happy       1642
sad          460
angry        143
Name: count, dtype: int64

Examples of classified sentiments:


,text,sentiment
4726,President Trump is saying he's the one,neutral
1720,the Office of Secretary of Defense with,neutral
2158,the Secretary of Defense. We're going to,neutral
750,"look at that lineup. Oh boy. Now, we",neutral
18445,"I mean, do you some people argue \nthat the MA...",sad
14845,political repression. uh they,neutral
10337,is the United States looking to,neutral
423,I'm I'm still rather confused at why why,sad
16998,"Treasury Secretary, now that we've been",neutral
15047,gunfire with Venezuelan personnel,neutral


In [ ]:
import pandas as pd
from collections import Counter
import re

# Load the Excel file into a pandas DataFrame
try:
    df = pd.read_excel('/content/Extraction_Transcripts.xlsx')
    print('Excel file loaded successfully.')
except FileNotFoundError:
    print('Error: Extraction_Transcripts.xlsx not found. Please ensure the file is in the /content/ directory.')
    df = pd.DataFrame({'text': []}) # Create an empty DataFrame to avoid further errors

# Display the first few rows and column info to verify loading
display(df.head())
df.info()

Excel file loaded successfully.


,text,start,duration,video_id
0,Uh let's bring in constitutional,0.000,2.800,wRp33cQKoDA
1,attorney and former federal prosecutor,1.199,3.361,wRp33cQKoDA
2,"Katie Traskky. Katie, good morning to",2.800,3.519,wRp33cQKoDA
3,you. So there is some nuance to this,4.560,3.279,wRp33cQKoDA
4,Supreme Court ruling on the Alien,6.319,3.521,wRp33cQKoDA


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18480 entries, 0 to 18479
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   text      18480 non-null  object 
 1   start     18480 non-null  float64
 2   duration  18480 non-null  float64
 3   video_id  18480 non-null  object 
dtypes: float64(2), object(2)
memory usage: 577.6+ KB


In [ ]:
# Extract all text from the 'text' column and concatenate it into a single string
# Ensure the 'text' column exists and handle potential non-string types
if 'text' in df.columns:
    all_text = ' '.join(df['text'].dropna().astype(str).tolist())
else:
    all_text = ''
    print("Warning: 'text' column not found in the DataFrame.")

# Convert text to lowercase
all_text_lower = all_text.lower()

# Remove punctuation and numbers, keeping only alphabetic characters and spaces
cleaned_text = re.sub(r'[^a-z\s]', '', all_text_lower)

# Tokenize the cleaned text into words
words = cleaned_text.split()

# Create a corpus (list of all words) and count word frequencies
word_counts = Counter(words)

print(f"Total unique words found: {len(word_counts)}")
print(f"Total words (including repetitions): {len(words)}")

# Display the most common words
print("\nMost common words:")
for word, count in word_counts.most_common(100):
    print(f"'{word}': {count}")

Total unique words found: 9309
Total words (including repetitions): 118157

Most common words:
'the': 6048
'to': 3598
'and': 3517
'of': 2996
'that': 2675
'a': 2415
'is': 1976
'in': 1975
'you': 1729
'i': 1420
'this': 1334
'it': 1231
'we': 1074
'for': 1027
'on': 907
'have': 849
'are': 847
'they': 753
'was': 741
'so': 729
'what': 707
'not': 669
'but': 641
'with': 636
'its': 634
'uh': 628
'he': 600
'be': 568
'as': 568
'about': 562
'were': 532
'know': 520
'at': 504
'there': 479
'do': 458
'going': 456
'think': 445
'people': 444
'all': 416
'just': 416
'now': 413
'has': 412
'if': 407
'from': 389
'thats': 387
'president': 380
'because': 359
'an': 357
'our': 355
'like': 349
'been': 347
'or': 346
'who': 345
'out': 339
'these': 337
'right': 328
'by': 324
'trump': 322
'us': 319
'when': 313
'will': 304
'can': 302
'your': 302
'well': 301
'um': 301
'his': 288
'up': 283
'their': 276
'very': 270
'one': 267
'here': 262
'more': 260
'would': 247
'dont': 245
'how': 242
'get': 242
'them': 241
'want': 228
'so

In [ ]:
# Function to get contextual text by combining current text with neighbors
def get_contextual_text(df_series, idx, window_size=2):
    """Combines text from the current row and its neighbors."""
    texts = []
    # Add texts from previous rows within the window
    for i in range(max(0, idx - window_size), idx):
        if pd.notna(df_series.iloc[i]):
            texts.append(str(df_series.iloc[i]))
    # Add text from the current row
    if pd.notna(df_series.iloc[idx]):
        texts.append(str(df_series.iloc[idx]))
    # Add texts from next rows within the window
    for i in range(idx + 1, min(len(df_series), idx + 1 + window_size)):
        if pd.notna(df_series.iloc[i]):
            texts.append(str(df_series.iloc[i]))
    return ' '.join(texts).strip()

# Create the 'contextual_text' column
print("Generating 'contextual_text' column...")
df['contextual_text'] = [get_contextual_text(df['text'], i, window_size=2) for i in range(len(df))]
print("'contextual_text' column generated. Displaying head of new column:")
display(df[['text', 'contextual_text']].head())

Generating 'contextual_text' column...
'contextual_text' column generated. Displaying head of new column:


,text,contextual_text
0,Uh let's bring in constitutional,Uh let's bring in constitutional attorney and ...
1,attorney and former federal prosecutor,Uh let's bring in constitutional attorney and ...
2,"Katie Traskky. Katie, good morning to",Uh let's bring in constitutional attorney and ...
3,you. So there is some nuance to this,attorney and former federal prosecutor Katie T...
4,Supreme Court ruling on the Alien,"Katie Traskky. Katie, good morning to you. So ..."


In [ ]:
def classify_sentiment_with_context(contextual_text):
    if pd.isna(contextual_text) or not str(contextual_text).strip():
        return 'neutral'

    analysis = TextBlob(str(contextual_text))
    polarity = analysis.sentiment.polarity
    subjectivity = analysis.sentiment.subjectivity

    # Adjusted Heuristic for broader 'angry' detection in grouped chunks
    if polarity > 0.15:
        return 'happy'
    elif polarity < -0.3 and subjectivity > 0.4:
        # Lowered polarity from -0.7 to -0.3 and subjectivity from 0.7 to 0.4
        return 'angry'
    elif polarity < -0.1:
        return 'sad'
    else:
        return 'neutral'

# Re-apply the updated contextual sentiment analysis function to the grouped dataframe
print("Re-applying adjusted contextual sentiment analysis to grouped data...")
agg_df['sentiment'] = agg_df['contextual_text'].apply(classify_sentiment_with_context)

# Display the updated distribution
sentiment_counts_agg = agg_df['sentiment'].value_counts()
print("\nUpdated Sentiment Distribution (Grouped Data):")
print(sentiment_counts_agg)

# Show samples of the new 'angry' classifications
print("\nSamples of 'angry' classifications in grouped data:")
display(agg_df[agg_df['sentiment'] == 'angry'].head(10))

Re-applying adjusted contextual sentiment analysis to grouped data...

Updated Sentiment Distribution (Grouped Data):
sentiment
neutral    1630
happy      1103
sad         287
angry       100
Name: count, dtype: int64

Samples of 'angry' classifications in grouped data:


,video_id,group_id,contextual_text,start,duration,sentiment
32,1CMC97YyvHY,2607,this country I hope are against uh politically...,441.599,23.601,angry
57,1CMC97YyvHY,2632,celebrating the death. We can do two things at...,737.360,19.520,angry
63,1NzW3o8zFEc,1618,"journalist Karen How, who's closely reported o...",18.800,34.079,angry
109,1NzW3o8zFEc,1664,Kenyan workers had to read through reams of th...,674.880,31.201,angry
171,3y3_wvVYeyw,1502,stopping the application of that force. Has re...,167.734,9.609,angry
181,3y3_wvVYeyw,1512,that are going up towards these vessels. I mea...,263.096,11.344,angry
279,4pzZAHtxDck,843,"So, it's it's going to be a, a United States e...",533.132,8.041,angry
294,4pzZAHtxDck,858,And it was terrible to listen to the stress th...,695.127,9.176,angry
299,4pzZAHtxDck,863,for everyone around the world? They don't feel...,740.372,9.877,angry
362,4rJn0qvRQyU,3012,that have said the mRNA platform is critical \...,627.133,14.500,angry


In [ ]:
output_filename_grouped = 'Grouped_Transcripts_with_Sentiment.xlsx'
agg_df.to_excel(output_filename_grouped, index=False)
print(f"Grouped DataFrame successfully saved to '{output_filename_grouped}' in the /content/ directory.")

Grouped DataFrame successfully saved to 'Grouped_Transcripts_with_Sentiment.xlsx' in the /content/ directory.


In [ ]:
import numpy as np

# Create a grouping key for every 6 rows
df['group_id'] = np.arange(len(df)) // 6

# Aggregate the data
agg_df = df.groupby(['video_id', 'group_id']).agg({
    'text': lambda x: ' '.join(x.astype(str)),
    'start': 'first',
    'duration': 'sum'
}).reset_index()

# Rename the combined text column to contextual_text
agg_df.rename(columns={'text': 'contextual_text'}, inplace=True)

# Re-apply sentiment analysis on the newly grouped larger text chunks
agg_df['sentiment'] = agg_df['contextual_text'].apply(classify_sentiment_with_context)

print(f"Grouped {len(df)} entries into {len(agg_df)} chunks of up to 6 sentences each.")
display(agg_df.head())

# Show new sentiment distribution for the grouped data
print("\nSentiment Distribution (Grouped Context):")
print(agg_df['sentiment'].value_counts())

Grouped 18480 entries into 3120 chunks of up to 6 sentences each.


,video_id,group_id,contextual_text,start,duration,sentiment
0,1CMC97YyvHY,2575,President Trump has just announced on Fox News...,0.320,17.998,neutral
1,1CMC97YyvHY,2576,"Kirk, killing him has been caught. Trump said,...",8.559,34.322,neutral
2,1CMC97YyvHY,2577,Officials also released photos and video of th...,26.800,29.760,neutral
3,1CMC97YyvHY,2578,gunman is seen jumping from a roof on campus a...,41.440,29.040,neutral
4,1CMC97YyvHY,2579,"attacks on the political left, saying, quote, ...",55.920,25.199,happy



Sentiment Distribution (Grouped Context):
sentiment
neutral    1630
happy      1103
sad         287
angry       100
Name: count, dtype: int64


In [ ]:
# Create a new DataFrame with sentiment counts per video_id
sentiment_per_video_df = agg_df.groupby('video_id')['sentiment'].value_counts().unstack(fill_value=0)

print("Sentiment counts per Video ID:")
display(sentiment_per_video_df.head())

Sentiment counts per Video ID:


sentiment,angry,happy,neutral,sad
video_id,,,,
1CMC97YyvHY,2,16,30,13
1NzW3o8zFEc,2,29,39,3
1cptsxRJpIQ,0,3,15,2
3y3_wvVYeyw,2,22,41,4
4pzZAHtxDck,3,33,50,10


In [ ]:
output_filename_sentiment_per_video = 'Sentiment_Per_Video_ID.xlsx'
sentiment_per_video_df.to_excel(output_filename_sentiment_per_video, index=True)
print(f"DataFrame 'sentiment_per_video_df' successfully saved to '{output_filename_sentiment_per_video}' in the /content/ directory.")

DataFrame 'sentiment_per_video_df' successfully saved to 'Sentiment_Per_Video_ID.xlsx' in the /content/ directory.


In [ ]:
print('DataFrame `df` (first 5 rows with sentiment column):')
display(df.head())

DataFrame `df` (first 5 rows with sentiment column):


,text,start,duration,video_id,sentiment
0,Uh let's bring in constitutional,0.000,2.800,wRp33cQKoDA,neutral
1,attorney and former federal prosecutor,1.199,3.361,wRp33cQKoDA,neutral
2,"Katie Traskky. Katie, good morning to",2.800,3.519,wRp33cQKoDA,happy
3,you. So there is some nuance to this,4.560,3.279,wRp33cQKoDA,neutral
4,Supreme Court ruling on the Alien,6.319,3.521,wRp33cQKoDA,neutral


In [ ]:
print('\nSeries `sentiment_counts` (sentiment distribution):')
display(sentiment_counts)


Series `sentiment_counts` (sentiment distribution):


,count
sentiment,
neutral,16235
happy,1642
sad,460
angry,143


In [ ]:
print("Top 20 'angry' sentiment entries:")
display(df[df['sentiment'] == 'angry'].head(20))

Top 20 'angry' sentiment entries:


,text,start,duration,video_id,sentiment
211,"violent gang members, those violent",461.599,4.081,wRp33cQKoDA,angry
553,Congress is stupid and funds money to,1219.840,4.480,wRp33cQKoDA,angry
813,hysterical. He said yesterday he was,337.919,4.000,nONGAKED9Gs,angry
885,because people are paying an awful lot,472.240,3.519,nONGAKED9Gs,angry
910,>> I want these hearings to expose the ugly,537.040,5.760,nONGAKED9Gs,angry
1043,"tragic, horrible incident. But I know",144.879,4.961,XqI_sFSPbJQ,angry
1056,"violent crime, defend the homeland,",176.640,5.920,XqI_sFSPbJQ,angry
1062,and women here in the rank and file that,190.879,3.921,XqI_sFSPbJQ,angry
1210,out our priorities on violent crime and,505.520,3.200,XqI_sFSPbJQ,angry
1298,bunch of them are really bad criminals.,694.959,3.761,XqI_sFSPbJQ,angry


In [ ]:
from collections import Counter, defaultdict
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

# Ensure stop_words is defined
stop_words = set(stopwords.words('english'))

# Initialize a dictionary to hold word counts for each sentiment
sentiment_word_counts = defaultdict(Counter)

# Get all unique sentiment categories from agg_df
sentiment_categories = agg_df['sentiment'].unique()

# Process text for each sentiment category using agg_df
for sentiment_cat in sentiment_categories:
    # Filter agg_df for the current sentiment category
    sentiment_agg_df = agg_df[agg_df['sentiment'] == sentiment_cat]

    # Extract and clean contextual_text for this sentiment
    if 'contextual_text' in sentiment_agg_df.columns:
        all_text_sentiment = ' '.join(sentiment_agg_df['contextual_text'].dropna().astype(str).tolist())
    else:
        all_text_sentiment = ''

    all_text_sentiment_lower = all_text_sentiment.lower()
    cleaned_text_sentiment = re.sub(r'[^a-z\s]', '', all_text_sentiment_lower)
    words_sentiment = cleaned_text_sentiment.split()

    # Filter out stopwords
    words_sentiment_filtered = [word for word in words_sentiment if word not in stop_words]

    # Count words for this sentiment category
    sentiment_word_counts[sentiment_cat] = Counter(words_sentiment_filtered)

# Convert the dictionary of Counters into a DataFrame
# Create a list of dictionaries, one for each sentiment's word counts
data = []
for sentiment_cat, counts in sentiment_word_counts.items():
    for word, count in counts.items():
        data.append({'word': word, 'sentiment': sentiment_cat, 'count': count})

# Create the DataFrame from the list of dictionaries
word_sentiment_df = pd.DataFrame(data)

# Pivot the table to have words as index and sentiments as columns
word_sentiment_pivot = word_sentiment_df.pivot_table(index='word', columns='sentiment', values='count', fill_value=0)

# Display the first few rows of the new DataFrame
print("\nWord counts per sentiment category (using grouped data, with stopwords removed):")
display(word_sentiment_pivot.head())

# Display some basic info
print(f"Total unique words across all sentiments (using grouped data, with stopwords removed): {len(word_sentiment_pivot)}")


Word counts per sentiment category (using grouped data, with stopwords removed):


sentiment,angry,happy,neutral,sad
word,,,,
aable,0.0,1.0,0.0,0.0
aagua,0.0,1.0,1.0,0.0
aaragua,0.0,0.0,1.0,0.0
aaup,0.0,0.0,1.0,0.0
abandon,0.0,0.0,2.0,0.0


Total unique words across all sentiments (using grouped data, with stopwords removed): 9176


In [ ]:
output_filename_pivot_grouped_no_stopwords = 'word_sentiment_pivot_grouped_no_stopwords.xlsx'
word_sentiment_pivot.to_excel(output_filename_pivot_grouped_no_stopwords, index=True)
print(f"DataFrame 'word_sentiment_pivot' (based on grouped data, with stopwords removed) successfully saved to '{output_filename_pivot_grouped_no_stopwords}' in the /content/ directory.")

DataFrame 'word_sentiment_pivot' (based on grouped data, with stopwords removed) successfully saved to 'word_sentiment_pivot_grouped_no_stopwords.xlsx' in the /content/ directory.


In [ ]:
# Calculate the total count for each word across all sentiments
word_sentiment_pivot['total_count'] = word_sentiment_pivot.sum(axis=1)

# Define a frequency threshold to filter out very rare words (e.g., words appearing < 3 times in total)
frequency_threshold = 3

# Filter the DataFrame to keep only words above the frequency threshold
word_sentiment_pivot_filtered = word_sentiment_pivot[word_sentiment_pivot['total_count'] >= frequency_threshold].drop(columns=['total_count'])

# Display the first few rows of the filtered DataFrame
print(f"\nWord counts per sentiment category (grouped, stopwords removed, frequency filtered - total count >= {frequency_threshold}):")
display(word_sentiment_pivot_filtered.head())

# Display some basic info about the filtered DataFrame
print(f"Total unique words after frequency filtering: {len(word_sentiment_pivot_filtered)}")


Word counts per sentiment category (grouped, stopwords removed, frequency filtered - total count >= 3):


sentiment,angry,happy,neutral,sad
word,,,,
ability,1.0,4.0,13.0,4.0
able,0.0,45.0,15.0,1.0
abolish,0.0,2.0,2.0,0.0
abortion,1.0,5.0,5.0,1.0
abortions,0.0,3.0,5.0,0.0


Total unique words after frequency filtering: 3505


By removing words that appear very infrequently, we aim to clean up potential typos or truly nonsensical entries. This method helps to focus on more statistically significant terms while still trying to preserve proper nouns (which typically appear more than 1-2 times if they are relevant).

If you'd like to adjust the `frequency_threshold`, let me know!

In [ ]:
# Save the filtered DataFrame to a new Excel file
output_filename_pivot_filtered = 'word_sentiment_pivot_grouped_no_stopwords_filtered.xlsx'
word_sentiment_pivot_filtered.to_excel(output_filename_pivot_filtered, index=True)
print(f"DataFrame 'word_sentiment_pivot_filtered' successfully saved to '{output_filename_pivot_filtered}' in the /content/ directory.")

DataFrame 'word_sentiment_pivot_filtered' successfully saved to 'word_sentiment_pivot_grouped_no_stopwords_filtered.xlsx' in the /content/ directory.


In [ ]:
output_filename_pivot_grouped = 'word_sentiment_pivot_grouped.xlsx'
word_sentiment_pivot.to_excel(output_filename_pivot_grouped, index=True)
print(f"DataFrame 'word_sentiment_pivot' (based on grouped data) successfully saved to '{output_filename_pivot_grouped}' in the /content/ directory.")

DataFrame 'word_sentiment_pivot' (based on grouped data) successfully saved to 'word_sentiment_pivot_grouped.xlsx' in the /content/ directory.


In [ ]:
output_filename_pivot = 'word_sentiment_pivot.xlsx'
word_sentiment_pivot.to_excel(output_filename_pivot, index=True)
print(f"DataFrame 'word_sentiment_pivot' successfully saved to '{output_filename_pivot}' in the /content/ directory.")

DataFrame 'word_sentiment_pivot' successfully saved to 'word_sentiment_pivot.xlsx' in the /content/ directory.


In [ ]:
# Save the DataFrame with sentiment to a new Excel file
output_filename = 'Extraction_Transcripts_with_Sentiment.xlsx'
df.to_excel(output_filename, index=False)
print(f"DataFrame successfully saved to '{output_filename}' in the /content/ directory.")

DataFrame successfully saved to 'Extraction_Transcripts_with_Sentiment.xlsx' in the /content/ directory.


In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
# Define the list of English stopwords
# You can customize this list if needed
stop_words = set(stopwords.words('english'))

# Or for a commented list:
# stop_words = set([
# 'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don', "don't", 'should', "should've", 'now', 'd', 'll', 'm', 'o', 're', 've', 'y', 'ain', 'aren', "aren't", 'couldn', "couldn't", 'didn', "didn't", 'doesn', "doesn't", 'hadn', "hadn't", 'hasn', "hasn't", 'haven', "haven't", 'isn', "isn't", 'ma', 'mightn', "mightn't", 'mustn', "mustn't", 'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't", 'wasn', "wasn't", 'weren', "weren't", 'won', "won't", 'wouldn', "wouldn't" ])

# Filter out stopwords from the list of words
filtered_words = [word for word in words if word not in stop_words]

# Recalculate word counts with filtered words
filtered_word_counts = Counter(filtered_words)

print(f"Total unique words after stopword removal: {len(filtered_word_counts)}")
print(f"Total words after stopword removal (including repetitions): {len(filtered_words)}")

# Display the most common words after stopword removal
print("\nMost common words after stopword removal:")
for word, count in filtered_word_counts.most_common(20):
    print(f"'{word}': {count}")

Total unique words after stopword removal: 9176
Total words after stopword removal (including repetitions): 58524

Most common words after stopword removal:
'uh': 628
'know': 520
'going': 456
'think': 445
'people': 444
'thats': 387
'president': 380
'like': 349
'right': 328
'trump': 322
'us': 319
'well': 301
'um': 301
'one': 267
'would': 247
'dont': 245
'get': 242
'want': 228
'said': 208
'im': 208
